# PyTorch Dataset for AnnData

The `AnnDataset` class provides a PyTorch Dataset implementation for AnnData objects, enabling seamless integration with PyTorch DataLoaders and machine learning workflows.

**Key Features:**
- **Memory efficiency**: Works with both in-memory AnnData objects and streams from backed HDF5 data without loading into memory
- **Data transforms**: Applies custom transformations to loaded expression data
- **Multiprocessing safety**: Safe HDF5 handling following h5py best practices
- **Optimized batch loading**: Sorts indices for efficient sequential disk access

This notebook demonstrates:

1. **Basic Usage**: Creating `AnnDataset` and using it with PyTorch `DataLoader`
2. **Data Transformations**: Creating custom Transform classes for preprocessing
3. **Transform Composition**: Using `Compose` to chain multiple transforms
4. **Data Subsetting**: Creating train/test splits with realistic proportions
5. **Training Integration**: Complete PyTorch training loop with single-cell data
6. **Metadata Access**: Accessing observation metadata alongside expression data




## Setup and Imports


In [1]:
import numpy as np
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn, optim
from torch.utils.data import DataLoader

from anndata.experimental.pytorch import AnnDataset, Compose, Transform

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


## Basic Usage

### Creating a Dataset


In [2]:
# Load real single-cell data
adata = sc.datasets.pbmc3k_processed()

# Create a basic dataset
dataset = AnnDataset(adata)

print(f"Dataset length: {len(dataset)}")
print(f"Dataset shape: {dataset.shape}")

# Get a single sample
sample = dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Expression data shape: {sample['X'].shape}")


Dataset length: 2638
Dataset shape: (2638, 1838)
Sample keys: ['X', 'obs_n_genes', 'obs_percent_mito', 'obs_n_counts', 'obs_louvain']
Expression data shape: torch.Size([1838])


### Using with DataLoader


In [3]:
# Create a DataLoader for training
# Note: Using num_workers=0 for notebook compatibility
# In scripts, you can use num_workers > 0 with Transform classes
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0  # Use 0 for notebooks, >0 works in scripts
)

# Iterate through batches
for batch in dataloader:
    X_batch = batch["X"]  # Shape: (batch_size, n_vars)
    print(f"Batch shape: {X_batch.shape}")
    print(f"Batch range: {X_batch.min():.3f} to {X_batch.max():.3f}")
    break


Batch shape: torch.Size([32, 1838])
Batch range: -2.076 to 10.000


## Data Transformations

Transforms are applied to the expression data (`X`) after it's loaded from the AnnData object. The `transform` parameter accepts only Transform class instances that inherit from `Transform`. This unified API ensures all transforms work seamlessly with multiprocessing by design.

### Creating Custom Transform Classes

Transform classes receive the loaded expression data as a PyTorch tensor and return the transformed tensor:


In [4]:
class NormalizeRowSum(Transform):
    """Normalize counts to target sum per cell."""
    def __init__(self, target_sum=1e4):
        self.target_sum = target_sum

    def __call__(self, x):
        # Ensure positive values
        x = torch.clamp(x, min=0)
        # Normalize to target sum
        row_sum = torch.sum(x, dim=-1, keepdim=True) + 1e-8
        return x * (self.target_sum / row_sum)

    def __repr__(self):
        return f"NormalizeRowSum(target_sum={self.target_sum})"


class LogTransform(Transform):
    def __call__(self, x):
        return torch.log1p(x)


# Test the transforms
example_data = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
normalize = NormalizeRowSum(target_sum=10)
log_transform = LogTransform()

print("Original data:")
print(example_data)
print("\nAfter normalization:")
print(normalize(example_data))
print("\nAfter log transform:")
print(log_transform(normalize(example_data)))


Original data:
tensor([[1., 2., 3.],
        [4., 5., 6.]])

After normalization:
tensor([[1.6667, 3.3333, 5.0000],
        [2.6667, 3.3333, 4.0000]])

After log transform:
tensor([[0.9808, 1.4663, 1.7918],
        [1.2993, 1.4663, 1.6094]])


### Composing Transforms


In [5]:
# Compose transforms
training_transform = Compose([
    NormalizeRowSum(target_sum=1e4),
    LogTransform(),
])

print("Composed transform:")
print(training_transform)

# Create dataset with transforms
dataset_with_transforms = AnnDataset(adata, transform=training_transform)

# Test transformed data
sample = dataset_with_transforms[0]
print(f"\nTransformed sample X shape: {sample['X'].shape}")
print(f"Transformed sample X range: {sample['X'].min():.3f} to {sample['X'].max():.3f}")


Composed transform:
Compose(
    NormalizeRowSum(target_sum=10000.0)
    LogTransform()
)

Transformed sample X shape: torch.Size([1838])
Transformed sample X range: 0.000 to 5.640


## Working with Subsets

You can create datasets that use only a subset of observations (rows) from your AnnData object using the `obs_subset` parameter.

In [6]:
# Create train/test split (80/20) of observations (rows)
n_rows = len(adata)  # Total number of rows/observations
all_indices = np.arange(n_rows)
train_indices, test_indices = train_test_split(all_indices, test_size=0.2, random_state=42)

# Create datasets using subsets of observations
train_dataset = AnnDataset(adata, obs_subset=train_indices, transform=training_transform)  # Uses only training rows
test_dataset = AnnDataset(adata, obs_subset=test_indices, transform=training_transform)    # Uses only test rows

print(f"Training dataset: {len(train_dataset)} items ({len(train_dataset)/n_rows:.1%})")
print(f"Test dataset: {len(test_dataset)} items ({len(test_dataset)/n_rows:.1%})")

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"\nTrain batches: {len(train_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")


Training dataset: 2110 items (80.0%)
Test dataset: 528 items (20.0%)

Train batches: 66
Test batches: 17


## Integration with Training Loops

In [7]:
# Prepare labels for supervised learning (using cell types)
# Encode cell type labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(adata.obs['louvain'].values)
n_classes = len(label_encoder.classes_)

print(f"Cell types: {label_encoder.classes_}")
print(f"Number of classes: {n_classes}")
print(f"Label distribution: {np.bincount(encoded_labels)}")

# Add labels to the datasets (for demonstration)
train_labels = encoded_labels[train_indices]
test_labels = encoded_labels[test_indices]

print(f"\nTraining labels shape: {train_labels.shape}")
print(f"Test labels shape: {test_labels.shape}")


Cell types: ['B cells' 'CD14+ Monocytes' 'CD4 T cells' 'CD8 T cells' 'Dendritic cells'
 'FCGR3A+ Monocytes' 'Megakaryocytes' 'NK cells']
Number of classes: 8
Label distribution: [ 342  480 1144  316   37  150   15  154]

Training labels shape: (2110,)
Test labels shape: (528,)


In [8]:
# Create model, loss, optimizer
n_features = dataset.shape[1]  # n_vars
model = nn.Sequential(
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, n_classes)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Model architecture:
Sequential(
  (0): Linear(in_features=1838, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=64, out_features=8, bias=True)
)

Total parameters: 244,168


In [9]:
# Training loop
num_epochs = 3
model.train()

for epoch in range(num_epochs):
    total_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(train_dataloader):
        X_batch = batch["X"]
        # Get corresponding labels for this batch
        batch_start = batch_idx * train_dataloader.batch_size
        batch_end = min(batch_start + train_dataloader.batch_size, len(train_labels))
        y_batch = torch.tensor(train_labels[batch_start:batch_end], dtype=torch.long)

        # Handle last batch size mismatch
        if X_batch.shape[0] != y_batch.shape[0]:
            y_batch = y_batch[:X_batch.shape[0]]

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_dataloader)}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1}/{num_epochs} completed. Average Loss: {avg_loss:.4f}")
    print("-" * 50)

print("Training completed!")


Epoch 1/3, Batch 0/66, Loss: 2.1400
Epoch 1/3, Batch 10/66, Loss: 1.8399
Epoch 1/3, Batch 20/66, Loss: 1.6741
Epoch 1/3, Batch 30/66, Loss: 1.4808
Epoch 1/3, Batch 40/66, Loss: 1.4443
Epoch 1/3, Batch 50/66, Loss: 1.5275
Epoch 1/3, Batch 60/66, Loss: 1.7472
Epoch 1/3 completed. Average Loss: 1.6929
--------------------------------------------------
Epoch 2/3, Batch 0/66, Loss: 1.4838
Epoch 2/3, Batch 10/66, Loss: 1.7906
Epoch 2/3, Batch 20/66, Loss: 1.7818
Epoch 2/3, Batch 30/66, Loss: 1.4586
Epoch 2/3, Batch 40/66, Loss: 1.4638
Epoch 2/3, Batch 50/66, Loss: 1.4699
Epoch 2/3, Batch 60/66, Loss: 1.7247
Epoch 2/3 completed. Average Loss: 1.6563
--------------------------------------------------
Epoch 3/3, Batch 0/66, Loss: 1.4974
Epoch 3/3, Batch 10/66, Loss: 1.8158
Epoch 3/3, Batch 20/66, Loss: 1.6688
Epoch 3/3, Batch 30/66, Loss: 1.3494
Epoch 3/3, Batch 40/66, Loss: 1.5804
Epoch 3/3, Batch 50/66, Loss: 1.5212
Epoch 3/3, Batch 60/66, Loss: 1.7472
Epoch 3/3 completed. Average Loss: 1.654

## Accessing Metadata

The dataset includes observation metadata in the returned items:


In [10]:
# Item contains both expression data and metadata
item = dataset[0]
print(f"Available keys: {list(item.keys())}")

# Expression data
X = item["X"]
print(f"Expression data shape: {X.shape}")

# Check for metadata (if available)
metadata_keys = [key for key in item if key.startswith('obs_')]
print(f"Available metadata keys: {metadata_keys}")

# Access specific metadata if available
if "obs_louvain" in item:
    cell_cluster = item["obs_louvain"]
    print(f"Cell cluster: {cell_cluster}")

if "obs_n_genes" in item:
    n_genes = item["obs_n_genes"]
    print(f"Number of genes: {n_genes}")


Available keys: ['X', 'obs_n_genes', 'obs_percent_mito', 'obs_n_counts', 'obs_louvain']
Expression data shape: torch.Size([1838])
Available metadata keys: ['obs_n_genes', 'obs_percent_mito', 'obs_n_counts', 'obs_louvain']
Cell cluster: CD4 T cells
Number of genes: 781
